In [2]:
# Install required packages
# torch/torchvision are already preinstalled on Kaggle, so we only need PennyLane
!pip -q install pennylane

import torch
import torch.nn as nn
import torch.nn.functional as F
import pennylane as qml
import numpy as np
from torch.utils.data import DataLoader
import torchvision as tv
from torchvision import transforms
import os, glob, random
from collections import Counter
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm

print("✅ Dependencies installed!")
print(f"PyTorch version: {torch.__version__}")
print(f"PennyLane version: {qml.version()}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 69.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 70.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 102.3 MB/s eta 0:00:0000:0100:01
✅ Dependencies installed!
PyTorch version: 2.10.0+cu128
PennyLane version: 0.45.1


In [3]:
# On Kaggle, add the dataset via the notebook UI instead of the Kaggle API:
#   Right sidebar -> "+ Add Input" -> search "FruitNet Indian Fruits Dataset with Quality" -> Add
# It will be mounted read-only under /kaggle/input/ — no kaggle.json needed.

def find_dataset_root(marker_folder="Good Quality_Fruits"):
    """Walk /kaggle/input to find the folder that actually contains the class subfolders."""
    for dirpath, dirnames, filenames in os.walk("/kaggle/input"):
        if marker_folder in dirnames:
            return dirpath
    raise FileNotFoundError(
        "Couldn't find the dataset under /kaggle/input. Make sure you've added it via "
        "'+ Add Input' in the right sidebar first."
    )

root = find_dataset_root()
print(f"✅ Dataset found at: {root}")
!ls "{root}"


✅ Dataset found at: /kaggle/input/datasets/shashwatwork/fruitnet-indian-fruits-dataset-with-quality/Processed Images_Fruits
'Bad Quality_Fruits'  'Good Quality_Fruits'  'Mixed Qualit_Fruits'


In [4]:
# Dataset configuration
# `root` was already discovered in the previous cell.

# Class mapping
folder_to_target = {
    "Good Quality_Fruits": "good",
    "Mixed Qualit_Fruits": "moderate",
    "Bad Quality_Fruits": "bad"
}

target_to_idx = {"good": 0, "moderate": 1, "bad": 2}
class_names = ["good", "moderate", "bad"]

print(f"Dataset root: {root}")
print(f"Class mapping: {folder_to_target}")


Dataset root: /kaggle/input/datasets/shashwatwork/fruitnet-indian-fruits-dataset-with-quality/Processed Images_Fruits
Class mapping: {'Good Quality_Fruits': 'good', 'Mixed Qualit_Fruits': 'moderate', 'Bad Quality_Fruits': 'bad'}


In [5]:
# Find all images
exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def list_images(d):
    return [p for p in glob.glob(os.path.join(d, "**", "*"), recursive=True)
            if os.path.isfile(p) and os.path.splitext(p)[1].lower() in exts]

def index_images(root):
    items = []
    for cls_folder, mapped in folder_to_target.items():
        class_dir = os.path.join(root, cls_folder)
        if not os.path.isdir(class_dir):
            continue
        for p in list_images(class_dir):
            items.append((p, target_to_idx[mapped]))
    return items

# Index all images
items = index_images(root)
print(f"Total images found: {len(items)}")

# Check distribution
class_counts = Counter([y for _, y in items])
print("Per-class counts:", {class_names[k]: v for k, v in class_counts.items()})

# Create splits
random.seed(42)
random.shuffle(items)
n = len(items)
train_items = items[:int(0.7*n)]
val_items = items[int(0.7*n):int(0.85*n)]
test_items = items[int(0.85*n):]

print(f"Train: {len(train_items)}, Val: {len(val_items)}, Test: {len(test_items)}")


Total images found: 19526
Per-class counts: {'good': 11664, 'moderate': 1074, 'bad': 6788}
Train: 13668, Val: 2929, Test: 2929


In [6]:
# Transforms (no ToTensor since we use read_image)
tx_train = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ColorJitter(0.1, 0.1, 0.1, 0.05)
])

tx_eval = transforms.Compose([
    transforms.Resize((128, 128))
])

# Dataset class
class FruitQualityDataset(torch.utils.data.Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        p, y = self.items[i]
        img = tv.io.read_image(p).float() / 255.0
        if img.size(0) == 1:
            img = img.expand(3, *img.shape[1:])
        img = self.transform(img)
        return img, y

# Create dataloaders
dl_tr = DataLoader(FruitQualityDataset(train_items, tx_train),
                   batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
dl_va = DataLoader(FruitQualityDataset(val_items, tx_eval),
                   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
dl_te = DataLoader(FruitQualityDataset(test_items, tx_eval),
                   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print("✅ DataLoaders created!")

# Test batch
xb, yb = next(iter(dl_tr))
print(f"Batch shapes: {xb.shape}, {yb.shape}")


✅ DataLoaders created!
Batch shapes: torch.Size([32, 3, 128, 128]), torch.Size([32])


In [7]:
# Classical CNN - outputs 8 features (2^3 = 8 for AmplitudeEmbedding)
class Featurizer(nn.Module):
    def __init__(self, out_dim=8):  # Changed from 4 to 8 for amplitude embedding
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 16, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.head = nn.Linear(64, out_dim)

    def forward(self, x):
        return self.head(self.backbone(x).flatten(1))

print("✅ Featurizer defined (outputs 8 features for AmplitudeEmbedding)")


✅ Featurizer defined (outputs 8 features for AmplitudeEmbedding)


In [8]:
# Quantum circuit with AmplitudeEmbedding
# 8 features require 3 qubits (2^3 = 8)
n_qubits = 3  # Changed from 4 to 3
n_layers = 4
n_features = 8  # Must be 2^n_qubits

dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def qnode(inputs, weights):
    # AmplitudeEmbedding: normalize=True handles normalization automatically
    qml.AmplitudeEmbedding(features=inputs, wires=range(n_qubits),
                          normalize=True, pad_with=0.0)

    # Variational layers
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))

    # Measurements for 3-class classification
    return [qml.expval(qml.PauliZ(i)) for i in range(3)]

weight_shapes = {"weights": (n_layers, n_qubits, 3)}
qlayer = qml.qnn.TorchLayer(qnode, weight_shapes)

print(f"✅ Quantum circuit created with AmplitudeEmbedding")
print(f"   Qubits: {n_qubits}, Features: {n_features}, Layers: {n_layers}")


✅ Quantum circuit created with AmplitudeEmbedding
   Qubits: 3, Features: 8, Layers: 4


In [9]:
# Hybrid Quantum-Classical Model
class HybridVQC(nn.Module):
    def __init__(self):
        super().__init__()
        self.fe = Featurizer(out_dim=n_features)
        self.q = qlayer

    def forward(self, x):
        # Extract features (8-dimensional vector)
        features = self.fe(x)

        # No tanh needed - AmplitudeEmbedding handles normalization
        # Just ensure features are real-valued
        logits = self.q(features)

        return logits

model = HybridVQC()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Hybrid VQC model created on {device}")
print(f"   Total parameters: {total_params:,}")


✅ Hybrid VQC model created on cuda
   Total parameters: 24,140


In [10]:
# Training configuration
opt = torch.optim.Adam(model.parameters(), lr=2e-3, weight_decay=1e-4)

# --- Class-weighted loss, sqrt-dampened ---
# Full inverse-frequency weighting (total / (n_classes * count)) gave "moderate" ~5-6x
# weight and was too aggressive (93% -> 90.3% accuracy). Taking the square root softens
# that push while still correcting for the ~59/35/6 class imbalance.
class_counts = Counter([y for _, y in train_items])
total = sum(class_counts.values())
class_weights = torch.tensor(
    [(total / (len(class_names) * class_counts[i])) ** 0.5 for i in range(len(class_names))],
    dtype=torch.float32
).to(device)
print(f"Class weights ({class_names}): {class_weights.tolist()}")
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

# /kaggle/working/ is Kaggle's equivalent of Drive: a persistent output directory.
# Anything saved here appears in the notebook's "Output" tab and is kept when you
# commit ("Save Version"), so you can download it or reuse it as an input dataset later.
checkpoint_dir = "/kaggle/working/vqc_amplitude_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = f"{checkpoint_dir}/best_model.pt"

print(f"✅ Training setup complete")
print(f"   Optimizer: Adam (lr=2e-3)")
print(f"   Checkpoints: {checkpoint_path}")

Class weights (['good', 'moderate', 'bad']): [0.7463030219078064, 2.4484152793884277, 0.9816420078277588]
✅ Training setup complete
   Optimizer: Adam (lr=2e-3)
   Checkpoints: /kaggle/working/vqc_amplitude_checkpoints/best_model.pt


In [11]:
# Training function
def run_epoch(dl, train=True):
    model.train(train)
    total, correct, loss_sum = 0, 0, 0.0

    for x, y in tqdm(dl, desc="Training" if train else "Validating", leave=False):
        x, y = x.to(device), y.to(device)

        if train:
            opt.zero_grad(set_to_none=True)

        logits = model(x)
        loss = loss_fn(logits, y)

        if train:
            loss.backward()
            opt.step()

        loss_sum += loss.item() * x.size(0)
        total += x.size(0)
        correct += (logits.argmax(1) == y).sum().item()

    return loss_sum / total, correct / total

# Training loop
print("🚀 Starting training with AmplitudeEmbedding...")
best_va, patience, wait = 0.0, 6, 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(30):
    print(f"\nEpoch {epoch+1}/30")
    print("-" * 40)

    tr_loss, tr_acc = run_epoch(dl_tr, True)
    va_loss, va_acc = run_epoch(dl_va, False)

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)

    print(f"Train - Loss: {tr_loss:.4f}, Acc: {tr_acc:.3f}")
    print(f"Val   - Loss: {va_loss:.4f}, Acc: {va_acc:.3f}")

    if va_acc > best_va:
        best_va = va_acc
        wait = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': opt.state_dict(),
            'best_val_acc': best_va,
            'history': history
        }, checkpoint_path)
        print(f"✅ New best! Val acc: {best_va:.3f}")
    else:
        wait += 1
        print(f"Patience: {wait}/{patience}")

    if wait >= patience:
        print("Early stopping!")
        break

print(f"\n✅ Training complete! Best val acc: {best_va:.3f}")


🚀 Starting training with AmplitudeEmbedding...

Epoch 1/30
----------------------------------------


Train - Loss: 0.8089, Acc: 0.713
Val   - Loss: 0.6446, Acc: 0.814
✅ New best! Val acc: 0.814

Epoch 2/30
----------------------------------------


Train - Loss: 0.6651, Acc: 0.796
Val   - Loss: 0.5775, Acc: 0.833
✅ New best! Val acc: 0.833

Epoch 3/30
----------------------------------------


Train - Loss: 0.6246, Acc: 0.806
Val   - Loss: 0.5376, Acc: 0.865
✅ New best! Val acc: 0.865

Epoch 4/30
----------------------------------------


Train - Loss: 0.5902, Acc: 0.824
Val   - Loss: 0.5049, Acc: 0.880
✅ New best! Val acc: 0.880

Epoch 5/30
----------------------------------------


...

Train - Loss: 0.4189, Acc: 0.916
Val   - Loss: 0.3948, Acc: 0.932
Patience: 4/6

Epoch 27/30
----------------------------------------


Train - Loss: 0.4112, Acc: 0.919
Val   - Loss: 0.3855, Acc: 0.933
Patience: 5/6

Epoch 28/30
----------------------------------------


Train - Loss: 0.4052, Acc: 0.922
Val   - Loss: 0.3828, Acc: 0.945
✅ New best! Val acc: 0.945

Epoch 29/30
----------------------------------------


Train - Loss: 0.4094, Acc: 0.923
Val   - Loss: 0.3952, Acc: 0.937
Patience: 1/6

Epoch 30/30
----------------------------------------


Train - Loss: 0.4076, Acc: 0.921
Val   - Loss: 0.4264, Acc: 0.917
Patience: 2/6

✅ Training complete! Best val acc: 0.945


In [13]:
# Load best model
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Loaded best model (val acc: {checkpoint['best_val_acc']:.3f})")

# Test evaluation
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for x, y in tqdm(dl_te, desc="Testing"):
        x = x.to(device)
        logits = model(x)
        y_true.extend(y.numpy())
        y_pred.extend(logits.argmax(1).cpu().numpy())

# Calculate metrics
test_acc = np.mean(np.array(y_true) == np.array(y_pred))
cm = confusion_matrix(y_true, y_pred)

print(f"\n{'='*50}")
print("FINAL TEST RESULTS (AmplitudeEmbedding)")
print(f"{'='*50}")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"\nConfusion Matrix:")
print(cm)
print(f"\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Save final results
final_checkpoint = {
    'model_state_dict': model.state_dict(),
    'test_acc': test_acc,
    'confusion_matrix': cm.tolist(),
    'class_names': class_names,
    'model_config': {
        'n_qubits': n_qubits,
        'n_layers': n_layers,
        'n_features': n_features,
        'embedding': 'AmplitudeEmbedding'
    }
}

save_path = "/kaggle/working/vqc_amplitude_final.pt"
torch.save(final_checkpoint, save_path)
print(f"\n✅ Final model saved: {save_path}")


✅ Loaded best model (val acc: 0.945)


Testing: 100%|██████████| 46/46 [00:36<00:00,  1.25it/s]


FINAL TEST RESULTS (AmplitudeEmbedding)
Test Accuracy: 0.9348 (93.48%)

Confusion Matrix:
[[1654   50   37]
 [  24  133   10]
 [  54   16  951]]

Classification Report:
              precision    recall  f1-score   support

        good       0.95      0.95      0.95      1741
    moderate       0.67      0.80      0.73       167
         bad       0.95      0.93      0.94      1021

    accuracy                           0.93      2929
   macro avg       0.86      0.89      0.87      2929
weighted avg       0.94      0.93      0.94      2929


✅ Final model saved: /kaggle/working/vqc_amplitude_final.pt


In [ ]:
# Import all dependencies
import torch
import torch.nn as nn
import torch.nn.functional as F
import pennylane as qml
import numpy as np
from torchvision import transforms
import torchvision as tv
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ All libraries imported!")


In [ ]:
# On Kaggle there's no Drive-mount step. If you're continuing in the SAME session
# where you just trained, the checkpoint is already in /kaggle/working/.
# If you're in a NEW notebook, attach your earlier notebook's output as an input:
#   "+ Add Input" -> "Notebook Output Files" -> pick the version that has the .pt file
# then point model_path at e.g. /kaggle/input/<your-notebook-name>/vqc_amplitude_final.pt

model_path = "/kaggle/working/vqc_amplitude_final.pt"
checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)

# Extract configuration
n_qubits = checkpoint['model_config']['n_qubits']
n_layers = checkpoint['model_config']['n_layers']
n_features = checkpoint['model_config']['n_features']
class_names = checkpoint['class_names']
test_acc = checkpoint['test_acc']
cm = np.array(checkpoint['confusion_matrix'])

print("✅ Model checkpoint loaded!")
print(f"   Test Accuracy: {test_acc:.4f}")
print(f"   Classes: {class_names}")
print(f"   Qubits: {n_qubits}, Layers: {n_layers}, Features: {n_features}")


In [ ]:
# 1. Feature Extractor
class Featurizer(nn.Module):
    def __init__(self, out_dim=8):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 16, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.head = nn.Linear(64, out_dim)

    def forward(self, x):
        return self.head(self.backbone(x).flatten(1))

# 2. Quantum Circuit
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def qnode(inputs, weights):
    qml.AmplitudeEmbedding(features=inputs, wires=range(n_qubits),
                          normalize=True, pad_with=0.0)
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(3)]

weight_shapes = {"weights": (n_layers, n_qubits, 3)}
qlayer = qml.qnn.TorchLayer(qnode, weight_shapes)

# 3. Hybrid Model
class HybridVQC(nn.Module):
    def __init__(self):
        super().__init__()
        self.fe = Featurizer(out_dim=n_features)
        self.q = qlayer

    def forward(self, x):
        features = self.fe(x)
        return self.q(features)

# Create and load model
model = HybridVQC()
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"✅ Model restored and ready on {device}!")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check if history exists in checkpoint
if 'history' in checkpoint:
    # Plot training history
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    history = checkpoint['history']
    epochs_range = range(1, len(history['train_loss']) + 1)

    # Loss plot
    ax1.plot(epochs_range, history['train_loss'], 'b-', marker='o', label='Train Loss', linewidth=2)
    ax1.plot(epochs_range, history['val_loss'], 'r-', marker='s', label='Val Loss', linewidth=2)
    ax1.set_title('Loss vs Epochs (AmplitudeEmbedding)', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Loss', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    # Accuracy plot
    ax2.plot(epochs_range, [acc*100 for acc in history['train_acc']], 'b-', marker='o', label='Train Acc', linewidth=2)
    ax2.plot(epochs_range, [acc*100 for acc in history['val_acc']], 'r-', marker='s', label='Val Acc', linewidth=2)
    ax2.set_title('Accuracy vs Epochs (AmplitudeEmbedding)', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('/kaggle/working/vqc_amplitude_training.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("✅ Training history plot saved!")
# Confusion matrix heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'}, annot_kws={'fontsize': 14, 'fontweight': 'bold'})
plt.title(f'Confusion Matrix (Test Accuracy: {test_acc:.3f})', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Predicted Label', fontsize=13, fontweight='bold')
plt.ylabel('True Label', fontsize=13, fontweight='bold')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.savefig('/kaggle/working/vqc_amplitude_confusion.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Confusion matrix saved to Google Drive")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

# Confusion matrix is already loaded in 'cm' variable from checkpoint
# class_names is already defined: ["good", "moderate", "bad"]

print("="*60)
print("CONFUSION MATRIX ANALYSIS")
print("="*60)

# 1. Display raw confusion matrix
print("\nConfusion Matrix (Raw Counts):")
print(f"{'':>10}", end="")
for cls in class_names:
    print(f"{cls:>10}", end="")
print("\n" + "-"*40)

for i, actual_class in enumerate(class_names):
    print(f"{actual_class:>10}", end="")
    for j in range(len(class_names)):
        print(f"{cm[i,j]:>10}", end="")
    print()

print("\nNote: Rows = Actual, Columns = Predicted")

# 2. Per-class metrics
print("\n" + "="*60)
print("PER-CLASS PERFORMANCE")
print("="*60)

for i, class_name in enumerate(class_names):
    total = cm[i].sum()
    correct = cm[i, i]

    if total > 0:
        # True Positives, False Positives, False Negatives
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - tp - fp - fn

        accuracy = correct / total
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        print(f"\n{class_name.upper()}:")
        print(f"  Samples:      {total:>4}")
        print(f"  Correct:      {correct:>4} / {total:<4} = {accuracy:.1%}")
        print(f"  Precision:    {precision:.3f}")
        print(f"  Recall:       {recall:.3f}")
        print(f"  F1-Score:     {f1:.3f}")

        # Show main confusions
        misclassified = []
        for j in range(len(class_names)):
            if i != j and cm[i, j] > 0:
                misclassified.append((class_names[j], cm[i, j], cm[i, j]/total*100))

        if misclassified:
            print(f"  Confused as:")
            for confused_class, count, percent in sorted(misclassified, key=lambda x: -x[1]):
                print(f"    → {confused_class:>8}: {count:>3} ({percent:>5.1f}%)")

# 3. Overall metrics
print("\n" + "="*60)
print("OVERALL METRICS")
print("="*60)

total_samples = cm.sum()
correct_predictions = np.trace(cm)
overall_accuracy = correct_predictions / total_samples

print(f"Total Samples:        {int(total_samples)}")
print(f"Correct Predictions:  {int(correct_predictions)}")
print(f"Overall Accuracy:     {overall_accuracy:.4f} ({overall_accuracy*100:.2f}%)")

# Macro and weighted averages
precisions, recalls, f1s, supports = precision_recall_fscore_support(
    y_true=np.repeat(np.arange(len(class_names)), cm.sum(axis=1).astype(int)),
    y_pred=np.concatenate([np.repeat(j, cm[i,j].astype(int))
                          for i in range(len(class_names))
                          for j in range(len(class_names))]),
    labels=np.arange(len(class_names)),
    average=None
)

macro_precision = precisions.mean()
macro_recall = recalls.mean()
macro_f1 = f1s.mean()

weighted_precision = (precisions * supports).sum() / supports.sum()
weighted_recall = (recalls * supports).sum() / supports.sum()
weighted_f1 = (f1s * supports).sum() / supports.sum()

print(f"\nMacro Average:")
print(f"  Precision: {macro_precision:.3f}")
print(f"  Recall:    {macro_recall:.3f}")
print(f"  F1-Score:  {macro_f1:.3f}")

print(f"\nWeighted Average:")
print(f"  Precision: {weighted_precision:.3f}")
print(f"  Recall:    {weighted_recall:.3f}")
print(f"  F1-Score:  {weighted_f1:.3f}")

print("\n" + "="*60)


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import io, random

def predict_fruit_quality(image_path_or_pil, model, class_names, device):
    """Predict fruit quality with AmplitudeEmbedding VQC. Accepts a file path or a PIL Image."""
    transform = transforms.Compose([transforms.Resize((128, 128))])

    if isinstance(image_path_or_pil, str):
        image = Image.open(image_path_or_pil).convert('RGB')
    else:
        image = image_path_or_pil.convert('RGB')

    img_tensor = transform(tv.transforms.functional.to_tensor(image)).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(img_tensor)
        probs = F.softmax(logits, dim=1).squeeze()
        pred_idx = logits.argmax(1).item()

    return image, {
        'predicted_class': class_names[pred_idx],
        'confidence': probs[pred_idx].item(),
        'all_probabilities': {class_names[i]: prob.item()
                            for i, prob in enumerate(probs)}
    }

def show_prediction(image, result):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.imshow(image)
    ax1.set_title('Input Image', fontsize=13, fontweight='bold')
    ax1.axis('off')

    probs = [result['all_probabilities'][c] for c in class_names]
    colors = ['green' if c == result['predicted_class'] else 'lightgray' for c in class_names]
    bars = ax2.bar(class_names, probs, color=colors, edgecolor='black', linewidth=1.5, alpha=0.8)
    ax2.set_title(f"Prediction: {result['predicted_class'].upper()}\nConfidence: {result['confidence']:.1%}",
                  fontsize=14, fontweight='bold',
                  color='darkgreen' if result['confidence'] > 0.7 else 'darkorange')
    ax2.set_ylabel('Probability', fontsize=12, fontweight='bold')
    ax2.set_ylim(0, 1.1)
    ax2.grid(axis='y', alpha=0.3, linestyle='--')
    for bar, prob in zip(bars, probs):
        ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.03,
                  f'{prob:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.show()

    print(f"\n{'='*50}")
    print(f"🍎 PREDICTION: {result['predicted_class'].upper()}")
    print(f"🎯 CONFIDENCE: {result['confidence']:.1%}")
    print(f"{'='*50}")

# --- Option A: interactive upload widget (works when editing the notebook live) ---
try:
    import ipywidgets as widgets
    from IPython.display import display

    uploader = widgets.FileUpload(accept='image/*', multiple=False)
    display(uploader)

    def on_upload_change(change):
        if uploader.value:
            files_dict = uploader.value
            file_info = list(files_dict.values())[0] if isinstance(files_dict, dict) else files_dict[0]
            content = file_info['content'] if isinstance(file_info, dict) else file_info.content
            image = Image.open(io.BytesIO(content))
            img, result = predict_fruit_quality(image, model, class_names, device)
            show_prediction(img, result)

    uploader.observe(on_upload_change, names='value')
    print("📤 Upload a fruit image above to test it (interactive editor sessions only).")
except ImportError:
    print("ipywidgets not available in this environment.")

# --- Option B: fallback demo on a random test-set image (works in any run mode) ---
sample_path, sample_label = random.choice(test_items)
img, result = predict_fruit_quality(sample_path, model, class_names, device)
print(f"\n(Fallback demo on a sample test image — true label: {class_names[sample_label]})")
show_prediction(img, result)


In [1]:
import torch
import numpy as np

model_path = "/kaggle/input/notebooks/naveenan05/hybrid-qml-fruit-quality/vqc_amplitude_final.pt"
checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)

print("Keys in checkpoint:", list(checkpoint.keys()))
print()

test_acc = checkpoint['test_acc']
cm = np.array(checkpoint['confusion_matrix'])
class_names = checkpoint['class_names']

print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Classes: {class_names}")
print(f"\nConfusion Matrix:\n{cm}")

# Derive per-class precision/recall/f1 directly from the confusion matrix
# (rows = true label, columns = predicted label)
print(f"\n{'Class':<12}{'Precision':<12}{'Recall':<12}{'F1-score':<12}{'Support':<10}")
print("-" * 58)

for i, cls in enumerate(class_names):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    support = cm[i, :].sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    print(f"{cls:<12}{precision:<12.2f}{recall:<12.2f}{f1:<12.2f}{support:<10}")

Keys in checkpoint: ['model_state_dict', 'test_acc', 'confusion_matrix', 'class_names', 'model_config']

Test Accuracy: 0.9191 (91.91%)
Classes: ['good', 'moderate', 'bad']

Confusion Matrix:
[[1692   13   36]
 [  84   63   20]
 [  80    4  937]]

Class       Precision   Recall      F1-score    Support   
----------------------------------------------------------
good        0.91        0.97        0.94        1741      
moderate    0.79        0.38        0.51        167       
bad         0.94        0.92        0.93        1021      
